In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-heart-murmur-detection\\reseach'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-heart-murmur-detection'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    # Directories
    root_dir: Path
    transformed_data_path: Path
    trained_model_path: Path

    # Model Architecture
    input_size: int
    hidden_size: int
    num_layers: int
    num_classes: int
    dropout: float

    # Training Hyperparameters
    batch_size: int
    learning_rate: float
    epochs: int
    patience: int

In [6]:
from HeartBeat.constant import *
from HeartBeat.utils.common import read_yaml, create_directories

In [7]:
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer

        create_directories([config.root_dir])

        return ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            trained_model_path=Path(config.trained_model_path),
            transformed_data_path = Path(config.transformed_data_path),

            input_size=config.input_size,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            num_classes=config.num_classes,
            dropout=config.dropout,

            batch_size=config.batch_size,
            learning_rate=config.learning_rate,
            epochs=config.epochs,
            patience=config.patience,
        )

In [8]:
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch
from collections import Counter
import copy
import numpy as np

In [9]:
# Components
class HeartMurmurLSTM(nn.Module):
    def __init__(self, config: ModelTrainerConfig ):
        super(HeartMurmurLSTM, self).__init__()
        self.config = config
        
        self.lstm = nn.LSTM(
            input_size  = self.config.input_size,
            hidden_size = self.config.hidden_size,  # 64 hidden units
            num_layers  = self.config.num_layers,   # 1 layer
            batch_first = True,
            dropout     = config.dropout if config.num_layers > 1 else 0
        )

        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_size, 32),  # 32 instead of 64
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(32, config.num_classes),
            )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.classifier(out)
        return out
    
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
    
    def create_dataloaders(self, X_train, y_train, X_val, y_val, X_test, y_test):
        """Convert numpy arrays to PyTorch DataLoaders"""

        def to_tensor(X, y):
            X_t = torch.tensor(X, dtype=torch.float32)
            y_t = torch.tensor(y, dtype=torch.float32)
            return TensorDataset(X_t, y_t)

        train_loader = DataLoader(to_tensor(X_train, y_train), batch_size=self.config.batch_size, shuffle=True)
        val_loader   = DataLoader(to_tensor(X_val,   y_val),   batch_size=self.config.batch_size)
        test_loader  = DataLoader(to_tensor(X_test,  y_test),  batch_size=self.config.batch_size)

        return train_loader, val_loader, test_loader

    def load_transformed_data(self):
        path = self.config.transformed_data_path

        X_train = np.load(path / "X_train.npy")
        X_val   = np.load(path / "X_val.npy")
        X_test  = np.load(path / "X_test.npy")

        y_train = np.load(path / "y_train.npy")
        y_val   = np.load(path / "y_val.npy")
        y_test  = np.load(path / "y_test.npy")

        return X_train, X_val, X_test, y_train, y_val, y_test

    def compute_class_weights(self, y_train):
        """
        Compute class weights for imbalanced classification.  
        """
        # If one‑hot encoded → convert to integer labels
        if y_train.ndim > 1:
            labels = np.argmax(y_train, axis=1)
        else:
            labels = y_train

        class_counts = np.bincount(labels)
        total_samples = class_counts.sum()

        weights = torch.tensor(
            total_samples / class_counts,
            dtype=torch.float32
        )

        print(f"Class counts : {class_counts}")
        print(f"Class weights: {weights}")

        return weights


    def train_model(self, model, train_loader, val_loader, class_weights):
        """
        model:        HeartMurmurLSTM instance
        train_loader: training DataLoader
        val_loader:   validation DataLoader
        epochs:       number of training epochs
        lr:           learning rate
        class_weights: optional tensor of class weights
        patience:      early stopping patience (epochs without improvement)
        """
        device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model     = model.to(device)

        class_weights = class_weights.to(device)

        criterion = nn.CrossEntropyLoss(weight=class_weights)

        optimizer = torch.optim.Adam(model.parameters(), lr=self.config.learning_rate)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', patience=5, factor=0.5, verbose=True
        )

        history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

        # ── Early stopping state ───────────────────────────────────
        best_val_loss  = float('inf')
        best_model     = None
        epochs_no_improve = 0

        for epoch in range(self.config.epochs):
            # ── Training ──────────────────────────────────────────
            model.train()
            train_loss, train_correct, train_total = 0, 0, 0

            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch.argmax(dim=1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                train_loss += loss.item()
                preds = outputs.argmax(dim=1)
                labels = y_batch.argmax(dim=1)
                train_correct += (preds == labels).sum().item()
                train_total += len(labels)

            # ── Validation ────────────────────────────────────────
            model.eval()
            val_loss, val_correct, val_total = 0, 0, 0

            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch.argmax(dim=1))
                    val_loss += loss.item()
                    preds = outputs.argmax(dim=1)
                    labels = y_batch.argmax(dim=1)
                    val_correct += (preds == labels).sum().item()
                    val_total   += len(labels)

            # ── Metrics ───────────────────────────────────────────
            train_loss_avg = train_loss / len(train_loader)
            val_loss_avg = val_loss / len(val_loader)
            train_acc = train_correct / train_total * 100
            val_acc = val_correct / val_total * 100

            history["train_loss"].append(train_loss_avg)
            history["train_acc"].append(train_acc)
            history["val_loss"].append(val_loss_avg)
            history["val_acc"].append(val_acc)

            scheduler.step(val_loss_avg)

            print(f"Epoch [{epoch+1:02d}/{self.config.epochs}] "
                f"Train Loss: {train_loss_avg:.4f} | Train Acc: {train_acc:.2f}% | "
                f"Val Loss: {val_loss_avg:.4f} | Val Acc: {val_acc:.2f}% | "
                f"LR: {optimizer.param_groups[0]['lr']:.6f}")
            
            # ── Early stopping ────────────────────────────────────
            if val_loss_avg < best_val_loss:
                best_val_loss = val_loss_avg
                best_model    = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
                print(f"  ✓ Best model saved (val_loss: {best_val_loss:.4f})")
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= self.config.patience:
                    print(f"\n Early stopping at epoch {epoch+1} — no improvement for {self.config.patience} epochs")
                    break
        
        # ── Restore best model ────────────────────────────────────
        model.load_state_dict(best_model)
        print(f"\nTraining complete. Best Val Loss: {best_val_loss:.4f}")
        return model, history

In [10]:
import os
import numpy as np
import torch
import torch

In [12]:
try:
    # Load configuration
    config = configurationManager()
    trainer_config = config.get_model_trainer_config()

    # Skip training if model already exists
    if trainer_config.trained_model_path.exists():
        print("✓ Trained model already exists.")
        print("Skipping model training.")

    else:
        print("No trained model found.")
        print("Starting model training...")

        # Initialize trainer
        trainer = ModelTrainer(trainer_config)

        # Load transformed data
        X_train, X_val, X_test, y_train, y_val, y_test = (
            trainer.load_transformed_data()
        )

        # Compute class weights
        class_weights = trainer.compute_class_weights(y_train)

        # Create DataLoaders
        train_loader, val_loader, test_loader = trainer.create_dataloaders(
            X_train,
            y_train,
            X_val,
            y_val,
            X_test,
            y_test
        )

        # Build model
        model = HeartMurmurLSTM(trainer_config)

        # Train model
        model, history = trainer.train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            class_weights=class_weights
        )

        # Save best model
        torch.save(
            model.state_dict(),
            trainer_config.trained_model_path
        )

        print(f"✓ Model saved to {trainer_config.trained_model_path}")

except Exception as e:
    raise e

[2026-06-27 13:02:37,549: INFO: common: ymal file config\config.yaml loaded sucessfully]
[2026-06-27 13:02:37,552: INFO: common: ymal file params.yaml loaded sucessfully]
[2026-06-27 13:02:37,553: INFO: common: created directory at artifacts]
[2026-06-27 13:02:37,555: INFO: common: created directory at artifacts/model_training]
No trained model found.
Starting model training...
Class counts : [ 80 278 905]
Class weights: tensor([15.7875,  4.5432,  1.3956])


c:\Users\Sandeep\anaconda3\envs\heart\lib\site-packages\torch\optim\lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch [01/100] Train Loss: 1.1028 | Train Acc: 17.42% | Val Loss: 1.0938 | Val Acc: 24.82% | LR: 0.001000
  ✓ Best model saved (val_loss: 1.0938)
Epoch [02/100] Train Loss: 1.0915 | Train Acc: 28.03% | Val Loss: 1.0647 | Val Acc: 31.21% | LR: 0.001000
  ✓ Best model saved (val_loss: 1.0647)
Epoch [03/100] Train Loss: 1.0408 | Train Acc: 52.49% | Val Loss: 1.0055 | Val Acc: 63.12% | LR: 0.001000
  ✓ Best model saved (val_loss: 1.0055)
Epoch [04/100] Train Loss: 0.9068 | Train Acc: 61.05% | Val Loss: 0.7091 | Val Acc: 69.50% | LR: 0.001000
  ✓ Best model saved (val_loss: 0.7091)
Epoch [05/100] Train Loss: 0.7541 | Train Acc: 57.56% | Val Loss: 0.7144 | Val Acc: 69.50% | LR: 0.001000
Epoch [06/100] Train Loss: 0.6858 | Train Acc: 62.00% | Val Loss: 0.6351 | Val Acc: 49.65% | LR: 0.001000
  ✓ Best model saved (val_loss: 0.6351)
Epoch [07/100] Train Loss: 0.5864 | Train Acc: 59.70% | Val Loss: 0.6177 | Val Acc: 65.96% | LR: 0.001000
  ✓ Best model saved (val_loss: 0.6177)
Epoch [08/100] Tra